# Layer 3: two moods that can switch within a session

Layer 2 gave each session one hidden mood for all its tracks: *Engaged* (few skips) or *Browsing* (many skips). It explained most of the differences between sessions, but its predictive checks showed two things it could not reproduce:

- the **within-session gap**: in the real data a skip is followed by another skip more often than the session's own skip rate explains (0.10 in the real data, 0 in Layer 2),
- the **skip/play changes**: real sessions change from skip to play less often than Layer 2 predicts (2.74 against 3.95 per session). Skips come in runs.

In this notebook the mood can **switch** from one track to the next. Each track has its own hidden mood, and the mood of a track only depends on the mood of the track before: this is a hidden Markov model. If the switch probabilities are 0, we are back to Layer 2.

(the goal is to see whether adding switching beats Layer 2)

In [ ]:
import json
import os
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mood_switching as ms
import posterior as po
import session_types as st
from checks import (all_checks, average_log_score, load_sessions,
                    real_data_checks, split_users, to_matrix)

SEED = 2026
rng = np.random.default_rng(SEED)

N_DRAWS = 20_000     # importance sampling draws
DF = 4               # degrees of freedom of the Student-t proposal
N_POSTERIOR = 1000   # resampled posterior draws (predictive checks, test score)
N_STARTS = 10        # EM starting points

# Beta(2, 2) for theta_e, theta_b, p0, q_eb and q_be
# EM uses Beta(3, 3) to find the mode on the log-odds scale
PRIOR = ms.PRIOR
EM_PRIOR = po.log_odds_prior(PRIOR)

#make the dirs for the results
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

BLUE = "#00a10d"
ORANGE = "#aa3102"
GREY = "#a4a4a4"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "xtick.color": GREY,
    "ytick.color": GREY,
    "axes.labelcolor": GREY,
})

The model functions are in `mood_switching.py` and the general tools (importance sampling, Hessian, predictive checks) in `posterior.py`, used as they are in Layer 2.
We also import `session_types.py` (Layer 2) to compare the two models.

## 1. The data

We use the same sessions and the same training/test split as Layers 1 and 2 (split by user, seed 2026): 20% of the users are kept aside, only for the additional analysis at the end.

The results of Layers 1 and 2 are read from `results/layer1.json` and `results/layer2.json` for the comparisons.

In [ ]:
sessions = load_sessions()
train, test = split_users(sessions, seed=SEED)
train_skips, train_mask = to_matrix(train)
test_skips, test_mask = to_matrix(test)
train_lengths = train_mask.sum(axis=1)

with open("results/layer1.json") as file:
    layer1 = json.load(file)
with open("results/layer2.json") as file:
    layer2 = json.load(file)

print(f"Training: {len(train_skips)} sessions, {train_mask.sum()} tracks, skip rate {train['skip'].mean():.4f}")
print(f"Test: {len(test_skips)} sessions, {test_mask.sum()} tracks")
print(f"Session length: {train_lengths.min()} to {train_lengths.max()} tracks, mean {train_lengths.mean():.2f}")

We recall the two statistics that Layer 2 could not reproduce, read from `results/layer2.json`:

In [ ]:
layer2_checks = pd.DataFrame(layer2["predictive_checks"]).T
layer2_checks.loc[["within-session gap", "skip/play changes"],
                  ["real data", "simulated mean", "share at least as large"]].round(4)

The training data have 4314 sessions and 67 122 tracks, the same as in Layers 1 and 2 (skip rate around 0.42).
The test data is only used at the end for extra investigation.

The table shows what Layer 3 must fix: Layer 2 gives a within-session gap of 0 (real data 0.10) and too many skip/play changes (3.95 against 2.74).